## Planner first, trainer second

Every Kaggle session is an experiment, not a build-the-whole-AI day.

1. Fill `HYPOTHESIS` and `SUCCESS_IF` in the next cell.
2. First session: `MODE = "smoke"`, `TRACK = "A"`.
3. Read the sample. Write the result in `lab_log.md`.
4. Only then `MODE = "train"` or Track B.

# Student baseline LM — train many times on different data

You provide text → train a bit → look at the sample → change the data → train again from the last checkpoint.

That is all “knows what I taught it” means.

## How to run on Colab / Kaggle
1. Upload this notebook
2. Colab: Runtime → GPU T4. Kaggle: Settings → GPU + Internet ON
3. Edit the **next cell only**: `TRACK`, `DATA_PACK`, `RUN_NAME`, `RESUME_FROM`
4. Run all
5. Read the printed sample
6. Change `DATA_PACK` + `RUN_NAME`, set `RESUME_FROM` to the last run folder, Run all again

| Goal | TRACK | DATA_PACK | RESUME_FROM |
|---|---|---|---|
| First English run | A | english | None |
| Then teach Python | A | python | runs/A-english |
| First assistant run | B | mixed | None |
| Then only bugfixes | B | bugs | runs/B-mixed |


In [ ]:
# =========================
# EDIT THIS CELL EACH RUN
# =========================
MODE = "smoke"           # "smoke" = short Kaggle test | "train" = real session
TRACK = "A"              # "A" from-scratch tiny GPT   |  "B" LoRA on Qwen 0.5B
DATA_PACK = "english"    # "english" | "python" | "bugs" | "mixed"
RUN_NAME = "A-smoke-1"   # new folder every run
RESUME_FROM = None       # e.g. "runs/A-english" ; None = start fresh

# Write these BEFORE you press Run all. That is the planning step.
HYPOTHESIS = "Track A on english will drop loss below chance and the sample will contain common English words."
SUCCESS_IF = "loss falls across printed steps AND sample is not random symbols."

TRACK_A_STEPS = 80 if MODE == "smoke" else 300
TRACK_B_MAX_STEPS = 10 if MODE == "smoke" else 60
SEED = 42

print("PLAN")
print("  mode:      ", MODE)
print("  track:     ", TRACK)
print("  pack:      ", DATA_PACK)
print("  run:       ", RUN_NAME)
print("  resume:    ", RESUME_FROM)
print("  a_steps:   ", TRACK_A_STEPS)
print("  b_steps:   ", TRACK_B_MAX_STEPS)
print("  hypothesis:", HYPOTHESIS)
print("  success if:", SUCCESS_IF)



## What “train again on new data” does

The model is a pile of numbers (`state_dict`). Training moves those numbers a little so next-token guesses match the **current** pack.

```
run 1  DATA_PACK=english   RESUME_FROM=None              → runs/A-english/model.pt
run 2  DATA_PACK=python    RESUME_FROM=runs/A-english    → runs/A-python/model.pt
run 3  DATA_PACK=bugs      RESUME_FROM=runs/A-python     → runs/A-bugs/model.pt
```

Each run **starts from the last weights**, then sees only the new pack for `TRACK_A_STEPS` updates.

Two rules:

1. **Change `RUN_NAME` every time** or you overwrite the previous folder.
2. **New data overwrites old habits.** If run 2 is only Python, English prose gets worse. That is catastrophic forgetting. Mix packs (`mixed`) or replay a little old data when you add a new skill.

You inspect results by reading the sample printed at the end of the run, not by waiting for a magical “done learning everything” flag.


In [ ]:
import os, sys, math, time, json, random, platform
from pathlib import Path

os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def where_am_i():
    if os.path.isdir("/kaggle"):
        return "kaggle"
    if "google.colab" in sys.modules or os.path.isdir("/content"):
        return "colab"
    return "local"

ENV = where_am_i()
print("env:", ENV)

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
torch.manual_seed(SEED)

RUNS = Path("runs")
RUNS.mkdir(exist_ok=True)
SAVE_DIR = RUNS / RUN_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("this run saves to:", SAVE_DIR.resolve())
print("resume from:", RESUME_FROM)

print("SMOKE vs TRAIN:", MODE)
if MODE == "smoke":
    print("This is a wiring test. Loss should move a little. Do not judge Clyde from a smoke run.")


In [ ]:
# Load data from Kaggle dataset attached in notebook
from pathlib import Path

# After attaching dataset `clyde-training-data` in Kaggle Settings:
SFT_PATH = Path("/kaggle/input/clyde-training-data/sft.jsonl")
EVAL_PATH = Path("/kaggle/input/clyde-training-data/eval.jsonl")

if SFT_PATH.exists():
    print("SFT data found at:", SFT_PATH)
    
    import json
    
    def load_jsonl(path):
        rows = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows

    sft_rows = load_jsonl(SFT_PATH)
    eval_rows = load_jsonl(EVAL_PATH) if EVAL_PATH.exists() else []
    
    print(f"SFT rows: {len(sft_rows)}")
    print(f"Eval rows: {len(eval_rows)}")
    
    # Show first row format
    if sft_rows:
        first = sft_rows[0]
        print(f"First SFT row keys: {list(first.keys())}")
        print(f"  pack: {first.get('pack')}")
        print(f"  user (first 80 chars): {first.get('user', '')[:80]}")
        print(f"  assistant (first 80 chars): {first.get('assistant', '')[:80]}")
    
    if eval_rows:
        first_eval = eval_rows[0]
        print(f"First eval row keys: {list(first_eval.keys())}")


In [ ]:
# ---------------- data packs you can swap ----------------
PACKS = {
    "english": '''
The cat sat on the mat. The student wrote a clear sentence.
A paragraph should have one idea. Prefer short words when they work.
Rewrite sloppy English: "the loop it go too far" becomes "The loop goes too far."
Explain first, then give the corrected sentence.
Hello, Ada. Practice makes the next-character guesses better on this kind of prose.
''',
    "python": '''
def add(a, b):
    return a + b

def greet(name):
    return f"Hello, {name}."

def factorial(n):
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

class Counter:
    def __init__(self):
        self.n = 0
    def inc(self):
        self.n += 1
''',
    "bugs": '''
# bug: range(n) includes 0 so factorial becomes 0
def broken_factorial(n):
    result = 1
    for i in range(n):
        result *= i
    return result

# fix
def factorial(n):
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

# bug: subtracts instead of adding
def broken_add(a, b):
    return a - b

# fix
def add(a, b):
    return a + b
''',
}

PACKS["mixed"] = PACKS["english"] + "\n" + PACKS["python"] + "\n" + PACKS["bugs"]

if DATA_PACK not in PACKS:
    raise ValueError("DATA_PACK must be one of " + ", ".join(PACKS))

CORPUS = PACKS[DATA_PACK].strip()
print("DATA_PACK:", DATA_PACK, "| chars:", len(CORPUS))
print("--- pack preview ---")
print(CORPUS[:400])
print("-------------------")

# Track B uses the same pack names as chat pairs
SFT_PACKS = {
    "english": [
        {"user": "Write one sentence that explains what a paragraph is.",
         "assistant": "A paragraph is a group of sentences about one idea."},
        {"user": "Rewrite clearly: the loop it go too far",
         "assistant": "The loop goes too far."},
        {"user": "What does a topic sentence do?",
         "assistant": "It states the main idea of the paragraph so the reader knows what is coming."},
    ],
    "python": [
        {"user": "Write a Python function add(a, b) that returns the sum.",
         "assistant": "def add(a, b):\n    return a + b\n"},
        {"user": "Write greet(name) that returns Hello, name.",
         "assistant": "def greet(name):\n    return f\"Hello, {name}.\"\n"},
    ],
    "bugs": [
        {"user": "Language: python\nExpected: factorial(5) == 120\nBuggy code:\n\ndef factorial(n):\n    result = 1\n    for i in range(n):\n        result *= i\n    return result\n",
         "assistant": "- range(n) yields 0..n-1, so you multiply by 0\n- loop from 1 to n\n\ndef factorial(n):\n    result = 1\n    for i in range(1, n + 1):\n        result *= i\n    return result\n"},
        {"user": "Language: python\nExpected: add(2, 3) == 5\nBuggy code:\n\ndef add(a, b):\n    return a - b\n",
         "assistant": "- it subtracts\n- keep the same name\n\ndef add(a, b):\n    return a + b\n"},
    ],
}
SFT_PACKS["mixed"] = SFT_PACKS["english"] + SFT_PACKS["python"] + SFT_PACKS["bugs"]
print("SFT rows in this pack:", len(SFT_PACKS[DATA_PACK]))


## Track A — tiny GPT, resume-aware

Architecture stays frozen across runs. Only the **weights** and the **data** change. If you change `n_embd` / `n_layer` you cannot resume; start a new family of runs.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

if DEVICE == "cuda":
    N_EMBD, N_HEAD, N_LAYER, BLOCK, BATCH, LR = 128, 4, 4, 64, 16, 3e-4
else:
    N_EMBD, N_HEAD, N_LAYER, BLOCK, BATCH, LR = 64, 4, 2, 32, 8, 3e-4
DROPOUT = 0.1
HEAD_DIM = N_EMBD // N_HEAD

CHARS = (
    "\n\t "
    + "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    + "0123456789"
    + "()[]{}.,:;=_+-*/<>!?#\"'\\@|$%^&~`"
)
VOCAB = len(CHARS)
STOI = {ch: i for i, ch in enumerate(CHARS)}
ITOS = {i: ch for ch, i in STOI.items()}

def encode(s):
    return [STOI.get(ch, STOI[" "]) for ch in s]

def decode(ids):
    if isinstance(ids, torch.Tensor):
        ids = ids.tolist()
    return "".join(ITOS[int(i)] for i in ids)

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_attn = nn.Linear(N_EMBD, 3 * N_EMBD)
        self.c_proj = nn.Linear(N_EMBD, N_EMBD)
        self.drop = nn.Dropout(DROPOUT)
        self.register_buffer("mask", torch.tril(torch.ones(BLOCK, BLOCK)).view(1, 1, BLOCK, BLOCK))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(N_EMBD, dim=-1)
        q = q.view(B, T, N_HEAD, HEAD_DIM).transpose(1, 2)
        k = k.view(B, T, N_HEAD, HEAD_DIM).transpose(1, 2)
        v = v.view(B, T, N_HEAD, HEAD_DIM).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(HEAD_DIM)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.drop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.c_proj(y))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(N_EMBD), nn.LayerNorm(N_EMBD)
        self.attn = CausalSelfAttention()
        self.fc = nn.Linear(N_EMBD, 4 * N_EMBD)
        self.proj = nn.Linear(4 * N_EMBD, N_EMBD)
        self.drop = nn.Dropout(DROPOUT)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.drop(self.proj(F.gelu(self.fc(self.ln2(x)))))
        return x

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(VOCAB, N_EMBD)
        self.pos = nn.Embedding(BLOCK, N_EMBD)
        self.drop = nn.Dropout(DROPOUT)
        self.blocks = nn.ModuleList([Block() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.head = nn.Linear(N_EMBD, VOCAB, bias=False)
        self.head.weight = self.tok.weight
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.tok(idx) + self.pos(torch.arange(T, device=idx.device))[None, :, :])
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, VOCAB), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n_tokens, temperature=0.7):
        self.eval()
        for _ in range(n_tokens):
            logits, _ = self(idx[:, -BLOCK:])
            probs = F.softmax(logits[:, -1, :] / max(temperature, 1e-6), dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

CONFIG = dict(n_embd=N_EMBD, n_head=N_HEAD, n_layer=N_LAYER, block=BLOCK, vocab=VOCAB)
print("A-config", CONFIG)


In [ ]:
def make_stream(text):
    data = torch.tensor(encode(text * 40), dtype=torch.long)
    def batch():
        ix = torch.randint(0, len(data) - BLOCK - 1, (BATCH,))
        x = torch.stack([data[i:i+BLOCK] for i in ix])
        y = torch.stack([data[i+1:i+BLOCK+1] for i in ix])
        return x.to(DEVICE), y.to(DEVICE)
    return data, batch

def save_run(model, history):
    ckpt = SAVE_DIR / "model.pt"
    torch.save({"state": model.state_dict(), "config": CONFIG, "pack": DATA_PACK,
                "history": history}, ckpt)
    meta = {
        "track": "A", "run_name": RUN_NAME, "data_pack": DATA_PACK,
        "resume_from": RESUME_FROM, "steps_this_run": TRACK_A_STEPS,
        "final_loss": history[-1]["loss"] if history else None,
        "config": CONFIG,
    }
    (SAVE_DIR / "meta.json").write_text(json.dumps(meta, indent=2))
    print("saved", ckpt)

def load_if_resuming(model):
    if not RESUME_FROM:
        print("fresh weights")
        return
    path = Path(RESUME_FROM) / "model.pt"
    blob = torch.load(path, map_location=DEVICE)
    if blob.get("config") != CONFIG:
        raise ValueError("checkpoint config differs from this notebook; cannot resume. "
                         "Use the same n_embd/n_layer family.")
    model.load_state_dict(blob["state"])
    print("loaded weights from", path, "| previous pack:", blob.get("pack"))

def run_track_a():
    data, make_batch = make_stream(CORPUS)
    print("stream tokens:", len(data), "| chance-level loss ~", round(math.log(VOCAB), 3))
    model = TinyGPT().to(DEVICE)
    load_if_resuming(model)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    model.train()
    history = []
    t0 = time.time()
    for step in range(1, TRACK_A_STEPS + 1):
        x, y = make_batch()
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if step == 1 or step % 50 == 0 or step == TRACK_A_STEPS:
            rec = {"step": step, "loss": round(loss.item(), 4), "sec": round(time.time()-t0, 1)}
            history.append(rec)
            print(f"step {step:4d}  loss={rec['loss']:.4f}  {rec['sec']}s")
    save_run(model, history)

    probes = {
        "english": "The student wrote",
        "python": "def factorial",
        "bugs": "def factorial",
        "mixed": "def add",
    }
    prompt = probes[DATA_PACK]
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    print("\n=== SAMPLE after", RUN_NAME, "on pack", DATA_PACK, "===")
    print(decode(model.generate(idx, 160, temperature=0.7)[0]))
    print("=== end sample ===")
    print("Next run: set RESUME_FROM =", str(SAVE_DIR), "and pick a new DATA_PACK / RUN_NAME")
    return model

if TRACK == "A":
    a_model = run_track_a()
else:
    print("TRACK != A, skip tiny GPT")


## Track B — same idea, pretrained 0.5B + LoRA

`RESUME_FROM` should point at a previous `runs/B-...` folder that contains the LoRA adapter (`adapter_config.json`). Different data, same adapter, more steps.


In [ ]:
B_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def run_track_b():
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformers>=4.44", "datasets", "accelerate", "peft"])
    from datasets import Dataset
    from transformers import (AutoTokenizer, AutoModelForCausalLM,
                              DataCollatorForLanguageModeling, Trainer, TrainingArguments)
    from peft import LoraConfig, get_peft_model, PeftModel

    rows = SFT_PACKS[DATA_PACK]
    print("SFT rows:", len(rows))
    tok = AutoTokenizer.from_pretrained(B_MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    def to_text(ex):
        messages = [
            {"role": "system", "content": "You are a brief tutor for English and programming."},
            {"role": "user", "content": ex["user"]},
            {"role": "assistant", "content": ex["assistant"]},
        ]
        return tok.apply_chat_template(messages, tokenize=False)

    ds = Dataset.from_list([{"text": to_text(r)} for r in rows])

    def tokenize(batch):
        out = tok(batch["text"], truncation=True, max_length=512)
        out["labels"] = [ids[:] for ids in out["input_ids"]]
        return out

    ds = ds.map(tokenize, batched=True, remove_columns=["text"])

    base = AutoModelForCausalLM.from_pretrained(
        B_MODEL,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    if RESUME_FROM and (Path(RESUME_FROM) / "adapter_config.json").exists():
        model = PeftModel.from_pretrained(base, RESUME_FROM, is_trainable=True)
        print("resumed LoRA from", RESUME_FROM)
    else:
        lora = LoraConfig(
            r=8, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
        )
        model = get_peft_model(base, lora)
        print("fresh LoRA")
    model.print_trainable_parameters()
    model.gradient_checkpointing_enable()

    args = TrainingArguments(
        output_dir=str(SAVE_DIR / "hf-trainer"),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=TRACK_B_MAX_STEPS,
        learning_rate=2e-4,
        logging_steps=10,
        save_steps=TRACK_B_MAX_STEPS,
        fp16=DEVICE == "cuda",
        report_to=[],
        remove_unused_columns=False,
        warmup_steps=5,
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=ds,
        data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
    )
    trainer.train()
    model.save_pretrained(SAVE_DIR)
    tok.save_pretrained(SAVE_DIR)
    meta = {"track": "B", "run_name": RUN_NAME, "data_pack": DATA_PACK,
            "resume_from": RESUME_FROM, "steps_this_run": TRACK_B_MAX_STEPS,
            "base": B_MODEL}
    (SAVE_DIR / "meta.json").write_text(json.dumps(meta, indent=2))
    print("saved adapter to", SAVE_DIR)

    model.eval()
    probe = SFT_PACKS[DATA_PACK][0]["user"]
    messages = [
        {"role": "system", "content": "You are a brief tutor for English and programming."},
        {"role": "user", "content": probe},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=160, temperature=0.2, do_sample=True)
    print("\n=== SAMPLE after", RUN_NAME, "on pack", DATA_PACK, "===")
    print("PROMPT:", probe[:200])
    print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    print("=== end sample ===")
    print("Next run: RESUME_FROM =", str(SAVE_DIR))

if TRACK == "B":
    run_track_b()
else:
    print("TRACK != B, skip SFT")


## After each run

1. Read the `=== SAMPLE ===` block. That is the result.
2. Download the `runs/<RUN_NAME>/` folder (Colab file pane / Kaggle Output) if the session might die.
3. Plan the next cell values, for example:

```
# after A-english finished
TRACK = "A"
DATA_PACK = "python"
RUN_NAME = "A-python"
RESUME_FROM = "runs/A-english"
```

4. Run all again. Compare the new sample to the old one.

Add your own material by editing `PACKS` or `SFT_PACKS` in the data cell. Paste essays into `english`, paste your functions into `python`, paste broken + fixed pairs into `bugs`.

Kaggle: Save Version after a run you care about. Colab: copy `runs/` to Drive or it disappears with the runtime.
